# Loan Data Cleaning

Data cleaning pipeline for `loan_data_2007_2014.csv` (Lending Club loans, 2007–2014).

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "loan_data_2007_2014.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
df.head()

Rows: 466,285  |  Columns: 75
Memory usage: 779.9 MB


,Unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,...,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
0,0,1077501,1296599,5000,5000,4975.0,36 months,10.65,162.87,B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1077430,1314167,2500,2500,2500.0,60 months,15.27,59.83,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1077175,1313524,2400,2400,2400.0,36 months,15.96,84.33,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1076863,1277178,10000,10000,10000.0,36 months,13.49,339.31,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1075358,1311748,3000,3000,3000.0,60 months,12.69,67.79,B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Initial inspection

In [ ]:
missing_summary = (
    pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "dtype": df.dtypes.astype(str),
    })
    .sort_values("missing_pct", ascending=False)
)

print("Columns with any missing values:")
display(missing_summary[missing_summary["missing_count"] > 0])

Columns with any missing values:


,missing_count,missing_pct,dtype
max_bal_bc,466285,100.00,float64
open_rv_24m,466285,100.00,float64
inq_fi,466285,100.00,float64
open_rv_12m,466285,100.00,float64
il_util,466285,100.00,float64
mths_since_rcnt_il,466285,100.00,float64
total_bal_il,466285,100.00,float64
open_il_24m,466285,100.00,float64
open_il_12m,466285,100.00,float64
open_il_6m,466285,100.00,float64



Duplicate rows: 0
Duplicate loan ids: 0


## 2. Remove useless columns

Drop columns that contain no usable data:
- CSV export index column (`Unnamed: 0`)
- Columns that are 100% missing (no values at all)

In [3]:
# Drop CSV index artifact
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# Identify columns with zero non-null values
empty_columns = df.columns[df.isna().all()].tolist()
print(f"Removing {len(empty_columns)} empty columns (100% missing):")
print(empty_columns)

df = df.drop(columns=empty_columns)

print(f"\nShape after dropping empty columns: {df.shape}")

Removing 17 empty columns (100% missing):
['annual_inc_joint', 'dti_joint', 'verification_status_joint', 'open_acc_6m', 'open_il_6m', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'inq_fi', 'total_cu_tl', 'inq_last_12m']

Shape after dropping empty columns: (466285, 57)


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 466285 entries, 0 to 466284
Data columns (total 57 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           466285 non-null  int64  
 1   member_id                    466285 non-null  int64  
 2   loan_amnt                    466285 non-null  int64  
 3   funded_amnt                  466285 non-null  int64  
 4   funded_amnt_inv              466285 non-null  float64
 5   term                         466285 non-null  str    
 6   int_rate                     466285 non-null  float64
 7   installment                  466285 non-null  float64
 8   grade                        466285 non-null  str    
 9   sub_grade                    466285 non-null  str    
 10  emp_title                    438697 non-null  str    
 11  emp_length                   445277 non-null  str    
 12  home_ownership               466285 non-null  str    
 13  annual_inc

## 3. Standardize missing values

Treat blank strings and whitespace-only values as missing so counts and imputation are consistent.

In [5]:
text_columns = df.select_dtypes(include=["object", "string"]).columns

for col in text_columns:
    df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)

print(f"Standardized blanks to NaN in {len(text_columns)} text columns.")

Standardized blanks to NaN in 22 text columns.


## 4. Fix data types

Convert date-like and numeric columns to appropriate types where possible.

In [6]:
# Term is stored as " 36 months" / " 60 months" — keep the numeric part
df["term"] = df["term"].astype(str).str.extract(r"(\d+)").astype(float)

# Parse date columns (month-year format from Lending Club)
date_columns = [
    "issue_d",
    "earliest_cr_line",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d",
]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format="%b-%y", errors="coerce")

# Ensure id columns are integers
for col in ["id", "member_id"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

print("Updated dtypes:")
print(df.dtypes.value_counts())

Updated dtypes:
float64           30
str               16
datetime64[us]     5
int64              4
Int64              2
Name: count, dtype: int64


## 6. Final summary

In [7]:
final_missing = (
    pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
    })
    .query("missing_count > 0")
    .sort_values("missing_pct", ascending=False)
)

print("Remaining columns with missing values:")
display(final_missing)

print("\nCleaned dataset preview:")
display(df.head())

# `df` is ready for analysis / modeling

Remaining columns with missing values:


,missing_count,missing_pct
mths_since_last_record,403647,86.57
mths_since_last_major_derog,367311,78.77
desc,340539,73.03
mths_since_last_delinq,250351,53.69
next_pymnt_d,227214,48.73
tot_cur_bal,70276,15.07
tot_coll_amt,70276,15.07
total_rev_hi_lim,70276,15.07
emp_title,27588,5.92
emp_length,21008,4.51



Cleaned dataset preview:


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim
0,1077501,1296599,5000,5000,4975.0,36.0,10.65,162.87,B,B2,...,NaT,2016-01-01,0.0,NaN,1,INDIVIDUAL,0.0,NaN,NaN,NaN
1,1077430,1314167,2500,2500,2500.0,60.0,15.27,59.83,C,C4,...,NaT,2013-09-01,0.0,NaN,1,INDIVIDUAL,0.0,NaN,NaN,NaN
2,1077175,1313524,2400,2400,2400.0,36.0,15.96,84.33,C,C5,...,NaT,2016-01-01,0.0,NaN,1,INDIVIDUAL,0.0,NaN,NaN,NaN
3,1076863,1277178,10000,10000,10000.0,36.0,13.49,339.31,C,C1,...,NaT,2015-01-01,0.0,NaN,1,INDIVIDUAL,0.0,NaN,NaN,NaN
4,1075358,1311748,3000,3000,3000.0,60.0,12.69,67.79,B,B5,...,2016-02-01,2016-01-01,0.0,NaN,1,INDIVIDUAL,0.0,NaN,NaN,NaN


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 466285 entries, 0 to 466284
Data columns (total 57 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   id                           466285 non-null  Int64         
 1   member_id                    466285 non-null  Int64         
 2   loan_amnt                    466285 non-null  int64         
 3   funded_amnt                  466285 non-null  int64         
 4   funded_amnt_inv              466285 non-null  float64       
 5   term                         466285 non-null  float64       
 6   int_rate                     466285 non-null  float64       
 7   installment                  466285 non-null  float64       
 8   grade                        466285 non-null  str           
 9   sub_grade                    466285 non-null  str           
 10  emp_title                    438697 non-null  str           
 11  emp_length                   445277 n

## 7. Binary target label (`label`)

For default prediction we only keep loans with a **final** outcome. Ongoing or in-progress statuses are dropped because the label is not yet known.

| `loan_status` | Action |
|---|---|
| Current | **Remove** — loan still active |
| In Grace Period | **Remove** — still in early repayment |
| Late (16–30 days), Late (31–120 days) | **Remove** — delinquent but outcome not final |
| Fully Paid | **label = 1** (good) |
| Charged Off, Default | **label = 0** (bad) |
| Does not meet credit policy (Fully Paid / Charged Off) | **Keep** — rare policy exceptions with a resolved outcome |

In [11]:
# Statuses with unknown / in-progress outcome — drop before labeling
DROP_STATUSES = [
    "Current",
    "In Grace Period",
    "Late (16-30 days)",
]

rows_before = len(df)
df = df[~df["loan_status"].isin(DROP_STATUSES)].copy()
print(f"Dropped {rows_before - len(df):,} in-progress rows")
print(f"Remaining rows: {len(df):,}")
print(df["loan_status"].value_counts())

Dropped 228,590 in-progress rows
Remaining rows: 237,695
loan_status
Fully Paid                                             184739
Charged Off                                             42475
Late (31-120 days)                                       6900
Does not meet the credit policy. Status:Fully Paid       1988
Default                                                   832
Does not meet the credit policy. Status:Charged Off       761
Name: count, dtype: int64


In [13]:
# 1 = good debt (repaid), 0 = bad debt (default / charged off)
GOOD_STATUSES = [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid",
]
BAD_STATUSES = [
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off",
    "Late (31-120 days)"
]

status_to_label = {s: 1 for s in GOOD_STATUSES} | {s: 0 for s in BAD_STATUSES}
df["label"] = df["loan_status"].map(status_to_label)

unmapped = df["label"].isna().sum()
if unmapped:
    print("Unmapped statuses (should be 0):")
    print(df.loc[df["label"].isna(), "loan_status"].value_counts())
else:
    print("All remaining rows mapped to label.")

print("\nLabel distribution:")
print(df["label"].value_counts().rename({0: "bad (0)", 1: "good (1)"}))
print(f"\nBad rate: {(df['label'] == 0).mean():.2%}")

All remaining rows mapped to label.

Label distribution:
label
good (1)    186727
bad (0)      50968
Name: count, dtype: int64

Bad rate: 21.44%


In [15]:
df.info()

<class 'pandas.DataFrame'>
Index: 237695 entries, 0 to 466283
Data columns (total 58 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   id                           237695 non-null  Int64         
 1   member_id                    237695 non-null  Int64         
 2   loan_amnt                    237695 non-null  int64         
 3   funded_amnt                  237695 non-null  int64         
 4   funded_amnt_inv              237695 non-null  float64       
 5   term                         237695 non-null  float64       
 6   int_rate                     237695 non-null  float64       
 7   installment                  237695 non-null  float64       
 8   grade                        237695 non-null  str           
 9   sub_grade                    237695 non-null  str           
 10  emp_title                    224305 non-null  str           
 11  emp_length                   228539 non-nu

## 8. Feature engineering

Transform selected categorical / text columns into model-ready features.

In [19]:
df['title'].value_counts()

title
Debt consolidation         59800
Credit card refinancing    19424
Debt Consolidation         11310
Home improvement            5318
Other                       4877
                           ...  
Lawn care                      1
finish det payoff              1
Stop the bleeding              1
LoanGetter                     1
Consolidation 01               1
Name: count, Length: 49853, dtype: int64

In [ ]:
# desc → has_desc (1 if borrower wrote a description, else 0)
df["has_desc"] = df["desc"].notna().astype(int)
df = df.drop(columns=["desc"])

# emp_length → numeric years
# Max at 10+ years
def parse_emp_length(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text == "< 1 year":
        return 0.5
    if "+" in text:
        return float(text.split("+")[0])
    return float(text.split()[0])

df["emp_length"] = df["emp_length"].map(parse_emp_length)

# home_ownership: merge rare ANY into OTHER
df["home_ownership"] = (
    df["home_ownership"]
    .replace({"ANY": "OTHER"})
    .str.upper()
)

print("has_desc:", df["has_desc"].value_counts().to_dict())
print("emp_length sample:", df["emp_length"].describe()[["min", "50%", "max"]].to_dict())
print("home_ownership:\n", df["home_ownership"].value_counts())

has_desc: {0: 145958, 1: 91737}
emp_length sample: {'min': 0.5, '50%': 6.0, 'max': 10.0}
home_ownership:
 home_ownership
MORTGAGE    116769
RENT        100703
OWN          19995
OTHER          180
NONE            48
Name: count, dtype: int64


In [22]:
df['emp_length'].value_counts()

emp_length
10.0    70939
2.0     22246
0.5     19886
3.0     19295
5.0     17224
1.0     16006
4.0     15322
6.0     14242
7.0     13402
8.0     11042
9.0      8935
Name: count, dtype: int64

In [24]:
# emp_title → emp_pay_tier (low / medium / high) using job-title keywords
HIGH_PAY_KEYWORDS = [
    "doctor", "physician", "surgeon", "dentist", "lawyer", "attorney",
    "engineer", "software", "developer", "architect", "consultant",
    "director", "executive", "ceo", "cfo", "vice president", " vp",
    "manager", "professor", "pharmacist", "analyst", "pilot",
    "nurse practitioner", "financial adviser", "investment",
]
LOW_PAY_KEYWORDS = [
    "cashier", "clerk", "server", "waiter", "waitress", "barista",
    "janitor", "cleaner", "housekeep", "dishwash", "crew member",
    "retail", "sales associate", "stock clerk", "security guard",
    "landscap", "laborer", "warehouse", "picker", "packer",
    "helper", "intern", "student", "delivery driver", "fast food",
    "mcdonald", "burger king", "pizza hut", "subway", "starbucks",
]


def job_pay_tier(title):
    if pd.isna(title):
        return "none"
    text = str(title).lower()
    if any(keyword in text for keyword in HIGH_PAY_KEYWORDS):
        return "high"
    if any(keyword in text for keyword in LOW_PAY_KEYWORDS):
        return "low"
    return "medium"


df["emp_pay_tier"] = df["emp_title"].apply(job_pay_tier)
df = df.drop(columns=["emp_title"])

print(df["emp_pay_tier"].value_counts(dropna=False))

KeyError: 'emp_title'

### Other columns worth engineering next

| Column(s) | Idea |
|-----------|------|
| `grade` / `sub_grade` | Ordinal encode (A=1 … G=7); sub_grade adds finer risk band |
| `verification_status` | One-hot or ordinal (Not Verified < Source Verified < Verified) |
| `purpose` | One-hot or group rare purposes into `"other"` |
| `issue_d`, `earliest_cr_line` | Credit age: months between earliest credit line and loan issue |
| `annual_inc`, `loan_amnt`, `installment` | `installment_pct_inc` = installment / (annual_inc / 12) |
| `dti` | Already numeric; optional binning (low / medium / high) |
| `revol_util` | Cap above 100% (data errors), fill missing |
| `mths_since_last_delinq`, `mths_since_last_record` | Missing often means “never happened” — add flag + fill sentinel |
| `addr_state` | Region groups (West, South, …) or frequency encoding |
| `initial_list_status`, `application_type`, `pymnt_plan` | Binary / one-hot |
| `zip_code` | Keep first 3 digits only (already partially anonymized) |

**Drop before modeling (IDs, text, or target leakage):** `id`, `member_id`, `url`, `title`, `loan_status`, `out_prncp`, `out_prncp_inv`, `total_pymnt`, `total_pymnt_inv`, `total_rec_*`, `recoveries`, `collection_recovery_fee`, `last_pymnt_d`, `last_pymnt_amnt`, `next_pymnt_d`, `last_credit_pull_d`